<a href="https://colab.research.google.com/github/zmeikova/digitrecognition/blob/main/%D0%B3%D0%BF%D0%BE_%D0%B8%D1%82%D0%BE%D0%B3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('daily-website-visitors.csv')

# Чистим данные от запятых
df['Date'] = pd.to_datetime(df['Date'])
df['Unique.Visits'] = df['Unique.Visits'].str.replace(',', '').astype(int)


Берем только дату и посетителей

In [ ]:
data = df[['Date', 'Unique.Visits']].copy()
data = data.sort_values('Date')
data.set_index('Date', inplace=True)

print(f"Всего дней: {len(data)}")
print(data.head())

Лаги (предыдущие значения)

In [ ]:
data['lag_1'] = data['Unique.Visits'].shift(1)
data['lag_2'] = data['Unique.Visits'].shift(2)
data['lag_3'] = data['Unique.Visits'].shift(3)
data['lag_7'] = data['Unique.Visits'].shift(7)

Скользящие средние

In [ ]:
data['ma_7'] = data['Unique.Visits'].rolling(7).mean()
data['ma_14'] = data['Unique.Visits'].rolling(14).mean()

День недели

In [ ]:
data['day_of_week'] = data.index.dayofweek

Удаляем пустые строки

In [ ]:
data = data.dropna()

print(f"После создания признаков: {len(data)} строк")

80% на обучение, 20% на тест

In [ ]:
train_size = int(len(data) * 0.8)
train = data[:train_size]
test = data[train_size:]

Признаки и цель

In [ ]:
feature_cols = ['lag_1', 'lag_2', 'lag_3', 'lag_7', 'ma_7', 'ma_14', 'day_of_week']
X_train = train[feature_cols].values
y_train = train['Unique.Visits'].values
X_test = test[feature_cols].values
y_test = test['Unique.Visits'].values

 Нормализуем данные

In [ ]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1))
X_test_scaled = scaler_X.transform(X_test)
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1))

print(f"\nОбучающая выборка: {len(X_train_scaled)} строк")
print(f"Тестовая выборка: {len(X_test_scaled)} строк")


 Нужно переделать данные в 3D формат (samples, timesteps, features)

In [ ]:
def make_3d_data(X, y, timesteps=5):
    X_3d, y_3d = [], []
    for i in range(len(X) - timesteps):
        X_3d.append(X[i:i+timesteps])
        y_3d.append(y[i+timesteps])
    return np.array(X_3d), np.array(y_3d)

timesteps = 5
X_train_3d, y_train_3d = make_3d_data(X_train_scaled, y_train_scaled, timesteps)
X_test_3d, y_test_3d = make_3d_data(X_test_scaled, y_test_scaled, timesteps)

print(f"3D данные: обучение {X_train_3d.shape}, тест {X_test_3d.shape}")


Простая LSTM модель

In [ ]:
model = Sequential([
    LSTM(50, input_shape=(timesteps, X_train_3d.shape[2])),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')

Обучаем

In [ ]:
history = model.fit(X_train_3d, y_train_3d,
                    validation_data=(X_test_3d, y_test_3d),
                    epochs=30, batch_size=32, verbose=0)

print("Обучение завершено!")

LSTM прогноз

In [ ]:
y_pred_lstm_scaled = model.predict(X_test_3d, verbose=0)
y_pred_lstm = scaler_y.inverse_transform(y_pred_lstm_scaled)
y_test_clean = scaler_y.inverse_transform(y_test_3d)

 Превращаем в одномерные массивы

In [ ]:
y_test_clean = y_test_clean.flatten()
y_pred_lstm = y_pred_lstm.flatten()


SMA прогноз (простое скользящее среднее)

In [ ]:
sma_predictions = []
for i in range(len(y_test_clean)):
    if i < 7:
        sma_predictions.append(y_test_clean[i])  # первые дни берем факт
    else:
        sma_predictions.append(np.mean(y_test_clean[i-7:i]))
sma_predictions = np.array(sma_predictions)

 EMA прогноз (экспоненциальное сглаживание)

In [ ]:
ema_predictions = []
alpha = 0.3
ema = y_test_clean[0]
for i in range(len(y_test_clean)):
    if i == 0:
        ema_predictions.append(y_test_clean[0])
    else:
        ema = alpha * y_test_clean[i] + (1 - alpha) * ema
        ema_predictions.append(ema)
ema_predictions = np.array(ema_predictions)

СЧИТАЕМ ОШИБКИ

In [ ]:
def calc_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return mae, rmse, mape

mae_lstm, rmse_lstm, mape_lstm = calc_metrics(y_test_clean, y_pred_lstm)
mae_sma, rmse_sma, mape_sma = calc_metrics(y_test_clean, sma_predictions)
mae_ema, rmse_ema, mape_ema = calc_metrics(y_test_clean, ema_predictions)

print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ МОДЕЛЕЙ")
print("="*50)
print(f"LSTM: MAE={mae_lstm:.0f}, RMSE={rmse_lstm:.0f}, MAPE={mape_lstm:.1f}%")
print(f"SMA:  MAE={mae_sma:.0f}, RMSE={rmse_sma:.0f}, MAPE={mape_sma:.1f}%")
print(f"EMA:  MAE={mae_ema:.0f}, RMSE={rmse_ema:.0f}, MAPE={mape_ema:.1f}%")

ГРАФИК 1: LOSS (ОШИБКА ОБУЧЕНИЯ

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], 'b-', label='Ошибка на обучении', linewidth=2)
plt.plot(history.history['val_loss'], 'r-', label='Ошибка на проверке', linewidth=2)
plt.title(' Ошибка LSTM модели при обучении', fontsize=12, fontweight='bold')
plt.xlabel('Эпоха обучения')
plt.ylabel('Ошибка (MSE)')
plt.legend()
plt.grid(True, alpha=0.3)

 ГРАФИК 2: СРАВНЕНИЕ ПРОГНОЗО

In [ ]:
plt.subplot(1, 2, 2)
dates = test.index[timesteps:timesteps+len(y_test_clean)]

Берем только первые 100 точек для наглядности

In [ ]:
if len(dates) > 100:
    dates = dates[:100]
    y_test_clean_plot = y_test_clean[:100]
    y_pred_lstm_plot = y_pred_lstm[:100]
    sma_predictions_plot = sma_predictions[:100]
    ema_predictions_plot = ema_predictions[:100]
else:
    y_test_clean_plot = y_test_clean
    y_pred_lstm_plot = y_pred_lstm
    sma_predictions_plot = sma_predictions
    ema_predictions_plot = ema_predictions

plt.plot(dates, y_test_clean_plot, 'b-', label='Факт', linewidth=2)
plt.plot(dates, y_pred_lstm_plot, 'r--', label='LSTM прогноз', linewidth=2)
plt.plot(dates, sma_predictions_plot, 'g:', label='SMA прогноз', linewidth=2)
plt.plot(dates, ema_predictions_plot, 'm:', label='EMA прогноз', linewidth=2)
plt.title('Г Сравнение прогнозов моделей', fontsize=12, fontweight='bold')
plt.xlabel('Дата')
plt.ylabel('Количество посетителей')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
print("\n" + "="*70)
print("ТАБЛИЦА СРАВНЕНИЯ (первые 10 дней теста)")
print("="*70)

results_table = pd.DataFrame({
    'Дата': dates[:10].strftime('%Y-%m-%d'),
    'Факт': y_test_clean_plot[:10].astype(int),
    'LSTM': y_pred_lstm_plot[:10].astype(int),
    'SMA': sma_predictions_plot[:10].astype(int),
    'EMA': ema_predictions_plot[:10].astype(int)
})
print(results_table.to_string(index=False))


In [ ]:
print("\n" + "="*50)
print("ВЫВОД:")
print("="*50)
best_model = "LSTM" if rmse_lstm < min(rmse_sma, rmse_ema) else ("SMA" if rmse_sma < rmse_ema else "EMA")
print(f"Лучшая модель по RMSE: {best_model}")
if best_model == "LSTM":
    improvement = ((min(rmse_sma, rmse_ema) - rmse_lstm) / min(rmse_sma, rmse_ema)) * 100
    print(f"LSTM предсказывает точнее на {improvement:.1f}%")

print(f"\nРекомендация: используйте модель {best_model} для прогнозирования")

ГРАФИК ПРОГНОЗИРОВАНИЯ



In [ ]:
plt.figure(figsize=(16, 6))

 Подготовка данных для графика

In [ ]:
forecast_dates = test.index[timesteps:timesteps+len(y_test_clean)]
actual_values = y_test_clean
lstm_forecast = y_pred_lstm

Рисуем фактические данные (обучение + тест)

In [ ]:
plt.plot(data.index[-len(train)-timesteps:],
         data['Unique.Visits'].values[-len(train)-timesteps:],
         'b-', label='Исторические данные', linewidth=1.5, alpha=0.7)


Рсуием фактические  значения тестовой выборки

In [ ]:
plt.plot(forecast_dates, actual_values, 'b-', label='Фактические значения (тест)', linewidth=2)

 Рисуем прогноз LSTM

In [ ]:
plt.plot(forecast_dates, lstm_forecast, 'r--', label='Прогноз LSTM', linewidth=2.5)

Добавляем область неопределённости (±1 стандартное отклонение ошибки)

In [ ]:
error = actual_values - lstm_forecast
std_error = np.std(error)
plt.fill_between(forecast_dates,
                 lstm_forecast - std_error,
                 lstm_forecast + std_error,
                 color='red', alpha=0.2, label='±1σ ошибка прогноза')

 Оформление графика

In [ ]:
plt.title('Прогнозирование посещаемости сайта с помощью LSTM', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Дата', fontsize=11)
plt.ylabel('Количество уникальных посетителей', fontsize=11)
plt.legend(fontsize=10, loc='best')
plt.grid(True, alpha=0.3, linestyle='--')
plt.xticks(rotation=45, ha='right')


 Форматирование оси дат

In [ ]:
plt.gca().xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m-%d'))
plt.gca().xaxis.set_major_locator(plt.matplotlib.dates.AutoDateLocator(maxticks=10))

plt.tight_layout()
plt.show()

ПРОГНОЗ НА БУДУЩЕЕ

Берем последние известные данные для инициализации

In [ ]:
last_sequence = X_test_scaled[-1:].copy()  # последний пример из теста
future_predictions = []


In [ ]:
for _ in range(7):
    # Преобразуем в 3D формат
    input_seq = np.array([last_sequence[-timesteps:]])
    if input_seq.shape[1] < timesteps:
        # Если не хватает данных, дополняем
        pad_len = timesteps - input_seq.shape[1]
        input_seq = np.pad(input_seq, ((0,0), (pad_len,0), (0,0)), mode='edge')


Делаем прогноз

In [ ]:
pred_scaled = model.predict(input_seq, verbose=0)[0, 0]
    pred = scaler_y.inverse_transform([[pred_scaled]])[0, 0]
    future_predictions.append(pred)

Обновляем последовательность (сдвиг и добавление нового прогноза)

In [ ]:
new_row = last_sequence[-1].copy()
    new_row[0] = pred_scaled  # обновляем lag_1 и т.д.
    last_sequence = np.vstack([last_sequence, new_row])

Вывод прогноза

In [ ]:
last_date = forecast_dates[-1]
for i, pred in enumerate(future_predictions, 1):
    future_date = last_date + pd.Timedelta(days=i)
    print(f"{future_date.strftime('%Y-%m-%d')}: {pred:.0f} посетителей")